# Split dataset - train / val + background

Splits annotated images (Label Studio YOLO export) into train (80%) and val (20%),
then copies them into `data/processed/` in the Ultralytics-expected layout.

Background images (no dolphin, no annotation) are added to `data/processed/train/images/`
without any `.txt` file. YOLO treats unannotated images as hard negatives automatically.

In [7]:
import random
import shutil
from pathlib import Path

SEED = 42
VAL_RATIO = 0.2
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

PROJECT_ROOT = Path.cwd().parent
SRC_IMAGES = PROJECT_ROOT / "data" / "labels" / "images"
SRC_LABELS = PROJECT_ROOT / "data" / "labels" / "labels"
SRC_BACKGROUNDS = PROJECT_ROOT / "data" / "backgrounds"  # images with no annotation
PROCESSED = PROJECT_ROOT / "data" / "processed"

stems = sorted([
    p.stem for p in SRC_IMAGES.iterdir()
    if p.suffix in IMAGE_EXTENSIONS and (SRC_LABELS / (p.stem + ".txt")).exists()
])

print(f"Annotated pairs found: {len(stems)}")

if SRC_BACKGROUNDS.exists():
    bg_images = [p for p in SRC_BACKGROUNDS.iterdir() if p.suffix in IMAGE_EXTENSIONS]
    print(f"Background images found: {len(bg_images)}")
else:
    bg_images = []
    print("No background folder - create data/backgrounds/ and add unannotated images")

Annotated pairs found: 725
Background images found: 40


In [8]:
random.seed(SEED)
random.shuffle(stems)

n_val = int(len(stems) * VAL_RATIO)
val_set = set(stems[:n_val])
trn_set = set(stems[n_val:])

print(f"Train: {len(trn_set)}  |  Val: {len(val_set)}  |  Background: {len(bg_images)}")

# Ultralytics recommends 10% background images; aim for 20-30% when false positives dominate
bg_ratio = len(bg_images) / (len(trn_set) + len(bg_images)) if bg_images else 0
print(f"Background ratio in train: {bg_ratio:.1%}")

Train: 580  |  Val: 145  |  Background: 40
Background ratio in train: 6.5%


In [9]:
def find_image(src_dir, stem):
    for ext in IMAGE_EXTENSIONS:
        p = src_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None

def copy_split(stems_subset, split_name):
    img_dst = PROCESSED / split_name / "images"
    lbl_dst = PROCESSED / split_name / "labels"
    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)

    for stem in stems_subset:
        src_img = find_image(SRC_IMAGES, stem)
        shutil.copy(src_img, img_dst / src_img.name)
        shutil.copy(SRC_LABELS / f"{stem}.txt", lbl_dst / f"{stem}.txt")

    print(f"  {split_name}: {len(stems_subset)} annotated images copied")

def copy_backgrounds(bg_paths):
    img_dst = PROCESSED / "train" / "images"
    for p in bg_paths:
        shutil.copy(p, img_dst / p.name)
    print(f"  train backgrounds: {len(bg_paths)} images copied (no label file)")

if PROCESSED.exists():
    shutil.rmtree(PROCESSED)

copy_split(trn_set, "train")
copy_split(val_set, "val")

if bg_images:
    copy_backgrounds(bg_images)

print("Done.")

  train: 580 annotated images copied
  val: 145 annotated images copied
  train backgrounds: 40 images copied (no label file)
Done.
